# Corporación Favorita Grocery Sales Forecasting
## Part 3: Feature Engineering (Winner's Solution)

This notebook implements feature engineering based on the **1st place solution**.

**Winner's Key Insights:**
- Use only 2017 data for training
- Statistical features with different time windows are crucial
- Holidays and some key combinations were NOT useful

**Memory Optimization:**
- Process data in chunks
- Use efficient dtypes
- Clear memory frequently
- Only load necessary columns

## 1. Setup & Configuration

In [ ]:
import os
import gc
import time
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


In [ ]:
class Config:
    """Configuration based on winner's solution"""
    
    # Paths
    INPUT_PATH = Path('/kaggle/input/favorita-grocery-sales-forecasting')
    OUTPUT_PATH = Path('/kaggle/working')
    
    # Date ranges (Winner's approach: only 2017 data)
    # Training windows
    TRAIN_START_1 = '2017-05-31'  # Option 1: Longer training
    TRAIN_START_2 = '2017-06-14'  # Option 2: Shorter training
    TRAIN_END = '2017-07-19'
    
    # Validation window
    VAL_START = '2017-07-26'
    VAL_END = '2017-08-10'
    
    # Test period (competition requirement: 16 days)
    TEST_START = '2017-08-16'
    TEST_END = '2017-08-31'
    
    # Feature extraction - need historical data before train start
    FEATURE_START = '2017-01-01'  # For 140-day lookback features
    
    # Time windows for statistical features (Winner's approach)
    NEAREST_DAYS = [1, 3, 5, 7, 14, 30, 60, 140]
    EQUAL_WINDOWS_1 = [1] * 16  # 16 windows of 1 day each
    EQUAL_WINDOWS_7 = [7] * 20  # 20 windows of 7 days each
    
    # Memory settings
    CHUNK_SIZE = 5_000_000
    
    RANDOM_STATE = 42

Config.OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
print(f"Feature extraction period: {Config.FEATURE_START} onwards")
print(f"Training period: {Config.TRAIN_START_1} to {Config.TRAIN_END}")
print(f"Validation period: {Config.VAL_START} to {Config.VAL_END}")

Feature extraction period: 2017-01-01 onwards
Training period: 2017-05-31 to 2017-07-19
Validation period: 2017-07-26 to 2017-08-10


## 2. Data Loading (Memory Efficient - 2017 Only)

In [ ]:
def load_2017_train_data():
    """
    Load only 2017 training data for memory efficiency.
    Winner's insight: Only 2017 data is useful for this competition.
    """
    print("\n" + "="*60)
    print("📊 LOADING 2017 TRAINING DATA")
    print("="*60)
    
    import subprocess
    
    # Get file path
    train_path = Config.INPUT_PATH / 'train.csv'
    if not train_path.exists():
        archive_path = Config.INPUT_PATH / 'train.csv.7z'
        if archive_path.exists():
            print("Extracting train.csv.7z...")
            subprocess.run(['7z', 'x', str(archive_path), f'-o{Config.OUTPUT_PATH}', '-y'],
                          capture_output=True)
            train_path = Config.OUTPUT_PATH / 'train.csv'
    
    # Define dtypes for memory efficiency
    dtypes = {
        'id': 'int32',
        'store_nbr': 'int8',
        'item_nbr': 'int32',
        'unit_sales': 'float32',
        'onpromotion': 'object'
    }
    
    print(f"\nLoading data from {Config.FEATURE_START} onwards...")
    start_time = time.time()
    
    # Load in chunks, filter to 2017 only
    chunks = []
    chunk_iter = pd.read_csv(
        train_path,
        dtype=dtypes,
        parse_dates=['date'],
        chunksize=Config.CHUNK_SIZE
    )
    
    feature_start = pd.to_datetime(Config.FEATURE_START)
    
    for i, chunk in enumerate(tqdm(chunk_iter, desc="Loading chunks")):
        # Filter to 2017 data only
        chunk = chunk[chunk['date'] >= feature_start]
        
        if len(chunk) > 0:
            # Process onpromotion
            chunk['onpromotion'] = chunk['onpromotion'].map(
                {'True': 1, 'False': 0, True: 1, False: 0}
            ).fillna(0).astype('int8')
            
            # Winner's preprocessing: fill negative unit_sales with 0
            chunk['unit_sales'] = chunk['unit_sales'].clip(lower=0)
            
            chunks.append(chunk)
        
        # Memory management
        if (i + 1) % 20 == 0:
            gc.collect()
    
    print("\nConcatenating chunks...")
    train = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    
    # Optimize dtypes
    train['store_nbr'] = train['store_nbr'].astype('int8')
    train['item_nbr'] = train['item_nbr'].astype('int32')
    train['unit_sales'] = train['unit_sales'].astype('float32')
    
    elapsed = time.time() - start_time
    print(f"\n✓ Loaded {len(train):,} rows in {elapsed:.1f}s")
    print(f"  Date range: {train['date'].min()} to {train['date'].max()}")
    print(f"  Memory: {train.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
    
    return train


def load_auxiliary_data():
    """Load stores, items, and other auxiliary data"""
    print("\n" + "="*60)
    print("📊 LOADING AUXILIARY DATA")
    print("="*60)
    
    import subprocess
    
    def get_path(filename):
        path = Config.INPUT_PATH / filename
        if not path.exists():
            archive = Config.INPUT_PATH / f"{filename}.7z"
            if archive.exists():
                subprocess.run(['7z', 'x', str(archive), f'-o{Config.OUTPUT_PATH}', '-y'],
                              capture_output=True)
                return Config.OUTPUT_PATH / filename
        return path
    
    # Load stores
    stores = pd.read_csv(get_path('stores.csv'))
    stores['store_nbr'] = stores['store_nbr'].astype('int8')
    stores['cluster'] = stores['cluster'].astype('int8')
    print(f"✓ Stores: {len(stores)} rows")
    
    # Load items
    items = pd.read_csv(get_path('items.csv'))
    items['item_nbr'] = items['item_nbr'].astype('int32')
    items['class'] = items['class'].astype('int16')
    items['perishable'] = items['perishable'].astype('int8')
    print(f"✓ Items: {len(items)} rows")
    
    # Load test data
    test = pd.read_csv(get_path('test.csv'), parse_dates=['date'])
    test['store_nbr'] = test['store_nbr'].astype('int8')
    test['item_nbr'] = test['item_nbr'].astype('int32')
    test['onpromotion'] = test['onpromotion'].map(
        {'True': 1, 'False': 0, True: 1, False: 0}
    ).fillna(0).astype('int8')
    print(f"✓ Test: {len(test)} rows")
    
    return stores, items, test

In [4]:
# Load data
train = load_2017_train_data()
stores, items, test = load_auxiliary_data()


📊 LOADING 2017 TRAINING DATA
Extracting train.csv.7z...

Loading data from 2017-01-01 onwards...


Loading chunks: 0it [00:00, ?it/s]


Concatenating chunks...

✓ Loaded 23,808,261 rows in 309.1s
  Date range: 2017-01-01 00:00:00 to 2017-08-15 00:00:00
  Memory: 0.49 GB

📊 LOADING AUXILIARY DATA
✓ Stores: 54 rows
✓ Items: 4100 rows
✓ Test: 3370464 rows


## 3. Create Base DataFrame for Features

In [5]:
def create_base_dataframe(train, test, stores, items):
    """
    Create base dataframe with category features.
    Winner's basic features: store, item, family, class, cluster
    """
    print("\n" + "="*60)
    print("🏗️ CREATING BASE DATAFRAME")
    print("="*60)
    
    # Get unique store-item combinations from train and test
    print("Getting unique store-item combinations...")
    
    train_pairs = train[['store_nbr', 'item_nbr']].drop_duplicates()
    test_pairs = test[['store_nbr', 'item_nbr']].drop_duplicates()
    all_pairs = pd.concat([train_pairs, test_pairs]).drop_duplicates()
    
    print(f"  Train pairs: {len(train_pairs):,}")
    print(f"  Test pairs: {len(test_pairs):,}")
    print(f"  Unique pairs: {len(all_pairs):,}")
    
    # Merge with store info
    print("\nMerging store information...")
    base = all_pairs.merge(stores[['store_nbr', 'type', 'cluster', 'state']], on='store_nbr')
    
    # Merge with item info
    print("Merging item information...")
    base = base.merge(items[['item_nbr', 'family', 'class', 'perishable']], on='item_nbr')
    
    # Create encoded features
    print("\nEncoding categorical features...")
    
    # Store type encoding
    type_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
    base['store_type'] = base['type'].map(type_map).astype('int8')
    
    # Family encoding
    family_map = {f: i for i, f in enumerate(base['family'].unique())}
    base['family_encoded'] = base['family'].map(family_map).astype('int8')
    
    # State encoding
    state_map = {s: i for i, s in enumerate(base['state'].unique())}
    base['state_encoded'] = base['state'].map(state_map).astype('int8')
    
    # Optimize dtypes
    base['store_nbr'] = base['store_nbr'].astype('int8')
    base['item_nbr'] = base['item_nbr'].astype('int32')
    base['cluster'] = base['cluster'].astype('int8')
    base['class'] = base['class'].astype('int16')
    base['perishable'] = base['perishable'].astype('int8')
    
    print(f"\n✓ Base dataframe created: {len(base):,} store-item pairs")
    print(f"  Columns: {list(base.columns)}")
    
    return base, family_map, state_map

In [6]:
base_df, family_map, state_map = create_base_dataframe(train, test, stores, items)


🏗️ CREATING BASE DATAFRAME
Getting unique store-item combinations...
  Train pairs: 167,515
  Test pairs: 210,654
  Unique pairs: 217,205

Merging store information...
Merging item information...

Encoding categorical features...

✓ Base dataframe created: 217,205 store-item pairs
  Columns: ['store_nbr', 'item_nbr', 'type', 'cluster', 'state', 'family', 'class', 'perishable', 'store_type', 'family_encoded', 'state_encoded']


## 4. Statistical Feature Functions (Winner's Approach)

In [7]:
def compute_lag_features(df, group_cols, target_col, date_col, ref_date, windows):
    """
    Compute statistical features for specified windows.
    
    Winner's approach:
    - Windows: [1,3,5,7,14,30,60,140]
    - Methods: mean, median, max, min, std
    
    Args:
        df: DataFrame with historical data
        group_cols: Columns to group by (e.g., ['store_nbr', 'item_nbr'])
        target_col: Target column for statistics
        date_col: Date column name
        ref_date: Reference date for calculating lags
        windows: List of window sizes in days
    
    Returns:
        DataFrame with lag features
    """
    ref_date = pd.to_datetime(ref_date)
    group_key = '_'.join(group_cols)
    
    # Filter data before reference date
    hist = df[df[date_col] < ref_date].copy()
    
    if len(hist) == 0:
        return None
    
    # Calculate days before reference
    hist['days_ago'] = (ref_date - hist[date_col]).dt.days
    
    features_list = []
    
    for window in windows:
        # Filter to window
        window_data = hist[hist['days_ago'] <= window]
        
        if len(window_data) == 0:
            continue
        
        # Compute statistics
        stats = window_data.groupby(group_cols)[target_col].agg([
            'mean', 'std', 'min', 'max', 'median', 'count'
        ]).reset_index()
        
        # Rename columns
        stats.columns = group_cols + [
            f'{group_key}_{target_col}_mean_{window}d',
            f'{group_key}_{target_col}_std_{window}d',
            f'{group_key}_{target_col}_min_{window}d',
            f'{group_key}_{target_col}_max_{window}d',
            f'{group_key}_{target_col}_median_{window}d',
            f'{group_key}_{target_col}_count_{window}d'
        ]
        
        features_list.append(stats)
    
    if not features_list:
        return None
    
    # Merge all windows
    result = features_list[0]
    for stats in features_list[1:]:
        result = result.merge(stats, on=group_cols, how='outer')
    
    return result


def compute_days_since_last(df, group_cols, date_col, ref_date):
    """
    Compute days since last appearance for each group.
    Winner's feature: days since last appearance
    """
    ref_date = pd.to_datetime(ref_date)
    hist = df[df[date_col] < ref_date].copy()
    
    if len(hist) == 0:
        return None
    
    # Get last date for each group
    last_dates = hist.groupby(group_cols)[date_col].max().reset_index()
    last_dates['days_since_last'] = (ref_date - last_dates[date_col]).dt.days
    
    group_key = '_'.join(group_cols)
    last_dates = last_dates.rename(columns={'days_since_last': f'{group_key}_days_since_last'})
    
    return last_dates[group_cols + [f'{group_key}_days_since_last']]


def compute_zero_ratio(df, group_cols, target_col, date_col, ref_date, window):
    """
    Compute ratio of zero sales in a window.
    Winner's target: zeros
    """
    ref_date = pd.to_datetime(ref_date)
    hist = df[df[date_col] < ref_date].copy()
    hist['days_ago'] = (ref_date - hist[date_col]).dt.days
    
    window_data = hist[hist['days_ago'] <= window]
    
    if len(window_data) == 0:
        return None
    
    # Compute zero ratio
    zero_stats = window_data.groupby(group_cols).apply(
        lambda x: (x[target_col] == 0).sum() / len(x)
    ).reset_index()
    
    group_key = '_'.join(group_cols)
    zero_stats.columns = group_cols + [f'{group_key}_zero_ratio_{window}d']
    
    return zero_stats

## 5. Memory-Efficient Feature Extraction

In [8]:
def extract_features_for_date(train, ref_date, base_df):
    """
    Extract all features for a single reference date.
    Memory-efficient: processes one date at a time.
    
    Winner's keys:
    - store x item (primary)
    - item only
    - store x class
    """
    ref_date = pd.to_datetime(ref_date)
    
    # Start with base features
    features = base_df.copy()
    
    # Filter historical data (only what we need)
    cutoff_date = ref_date - timedelta(days=150)  # Max window is 140 days + buffer
    hist = train[(train['date'] >= cutoff_date) & (train['date'] < ref_date)].copy()
    
    if len(hist) == 0:
        print(f"  ⚠ No historical data for {ref_date}")
        return features
    
    # === KEY 1: store x item ===
    print(f"    Computing store_item features...")
    
    # Lag features for unit_sales
    for window in [1, 3, 5, 7, 14, 30, 60, 140]:
        window_hist = hist[hist['date'] >= ref_date - timedelta(days=window)]
        if len(window_hist) > 0:
            stats = window_hist.groupby(['store_nbr', 'item_nbr'])['unit_sales'].agg([
                'mean', 'std', 'median'
            ]).reset_index()
            stats.columns = ['store_nbr', 'item_nbr', 
                           f'si_sales_mean_{window}d',
                           f'si_sales_std_{window}d', 
                           f'si_sales_median_{window}d']
            features = features.merge(stats, on=['store_nbr', 'item_nbr'], how='left')
        gc.collect()
    
    # Promotion features
    for window in [7, 14, 30]:
        window_hist = hist[hist['date'] >= ref_date - timedelta(days=window)]
        if len(window_hist) > 0:
            promo_stats = window_hist.groupby(['store_nbr', 'item_nbr'])['onpromotion'].agg(['mean', 'sum']).reset_index()
            promo_stats.columns = ['store_nbr', 'item_nbr',
                                  f'si_promo_mean_{window}d',
                                  f'si_promo_sum_{window}d']
            features = features.merge(promo_stats, on=['store_nbr', 'item_nbr'], how='left')
    
    # Days since last sale
    last_sale = hist.groupby(['store_nbr', 'item_nbr'])['date'].max().reset_index()
    last_sale['si_days_since_last'] = (ref_date - last_sale['date']).dt.days
    features = features.merge(last_sale[['store_nbr', 'item_nbr', 'si_days_since_last']], 
                             on=['store_nbr', 'item_nbr'], how='left')
    
    # Zero sales ratio
    for window in [7, 14, 30]:
        window_hist = hist[hist['date'] >= ref_date - timedelta(days=window)]
        if len(window_hist) > 0:
            zero_stats = window_hist.groupby(['store_nbr', 'item_nbr']).apply(
                lambda x: (x['unit_sales'] == 0).mean()
            ).reset_index()
            zero_stats.columns = ['store_nbr', 'item_nbr', f'si_zero_ratio_{window}d']
            features = features.merge(zero_stats, on=['store_nbr', 'item_nbr'], how='left')
    
    gc.collect()
    
    # === KEY 2: item only ===
    print(f"    Computing item features...")
    
    for window in [7, 14, 30, 60]:
        window_hist = hist[hist['date'] >= ref_date - timedelta(days=window)]
        if len(window_hist) > 0:
            item_stats = window_hist.groupby('item_nbr')['unit_sales'].agg([
                'mean', 'std', 'median'
            ]).reset_index()
            item_stats.columns = ['item_nbr',
                                 f'item_sales_mean_{window}d',
                                 f'item_sales_std_{window}d',
                                 f'item_sales_median_{window}d']
            features = features.merge(item_stats, on='item_nbr', how='left')
    
    gc.collect()
    
    # === KEY 3: store x class ===
    print(f"    Computing store_class features...")
    
    # Need to add class to historical data
    hist_with_class = hist.merge(items[['item_nbr', 'class']], on='item_nbr', how='left')
    
    for window in [7, 14, 30]:
        window_hist = hist_with_class[hist_with_class['date'] >= ref_date - timedelta(days=window)]
        if len(window_hist) > 0:
            sc_stats = window_hist.groupby(['store_nbr', 'class'])['unit_sales'].agg([
                'mean', 'median'
            ]).reset_index()
            sc_stats.columns = ['store_nbr', 'class',
                               f'sc_sales_mean_{window}d',
                               f'sc_sales_median_{window}d']
            features = features.merge(sc_stats, on=['store_nbr', 'class'], how='left')
    
    del hist_with_class
    gc.collect()
    
    return features

In [9]:
def create_training_features(train, test, base_df, date_range, include_target=True):
    """
    Create features for a range of dates.
    Memory-efficient batch processing.
    """
    print("\n" + "="*60)
    print(f"🔧 CREATING FEATURES FOR {len(date_range)} DATES")
    print("="*60)
    
    all_features = []
    
    for i, ref_date in enumerate(tqdm(date_range, desc="Processing dates")):
        print(f"\n  [{i+1}/{len(date_range)}] Processing {ref_date}...")
        
        # Extract features
        features = extract_features_for_date(train, ref_date, base_df)
        features['date'] = ref_date
        
        # Add day of week (Winner: only for some models)
        features['dayofweek'] = pd.to_datetime(ref_date).dayofweek
        
        # Add target if training data
        if include_target:
            day_data = train[train['date'] == ref_date][['store_nbr', 'item_nbr', 'unit_sales', 'onpromotion']]
            features = features.merge(day_data, on=['store_nbr', 'item_nbr'], how='left')
            features['unit_sales'] = features['unit_sales'].fillna(0)
            features['onpromotion'] = features['onpromotion'].fillna(0)
        else:
            # For test data, get promotion from test
            day_test = test[test['date'] == ref_date][['store_nbr', 'item_nbr', 'onpromotion', 'id']]
            features = features.merge(day_test, on=['store_nbr', 'item_nbr'], how='inner')
        
        all_features.append(features)
        
        # Memory management
        gc.collect()
    
    print("\nCombining all dates...")
    result = pd.concat(all_features, ignore_index=True)
    del all_features
    gc.collect()
    
    print(f"✓ Created features: {result.shape}")
    return result

## 6. Difference Features (Winner's Equal Windows)

In [10]:
def add_difference_features(df):
    """
    Add difference features between adjacent time windows.
    Winner's approach: difference of mean value between adjacent time windows
    """
    print("\n  Adding difference features...")
    
    # Differences for store_item sales
    if 'si_sales_mean_7d' in df.columns and 'si_sales_mean_14d' in df.columns:
        df['si_sales_diff_7_14'] = df['si_sales_mean_7d'] - df['si_sales_mean_14d']
    
    if 'si_sales_mean_14d' in df.columns and 'si_sales_mean_30d' in df.columns:
        df['si_sales_diff_14_30'] = df['si_sales_mean_14d'] - df['si_sales_mean_30d']
    
    if 'si_sales_mean_30d' in df.columns and 'si_sales_mean_60d' in df.columns:
        df['si_sales_diff_30_60'] = df['si_sales_mean_30d'] - df['si_sales_mean_60d']
    
    # Differences for item sales
    if 'item_sales_mean_7d' in df.columns and 'item_sales_mean_14d' in df.columns:
        df['item_sales_diff_7_14'] = df['item_sales_mean_7d'] - df['item_sales_mean_14d']
    
    if 'item_sales_mean_14d' in df.columns and 'item_sales_mean_30d' in df.columns:
        df['item_sales_diff_14_30'] = df['item_sales_mean_14d'] - df['item_sales_mean_30d']
    
    # Ratio features (relative change)
    if 'si_sales_mean_7d' in df.columns and 'si_sales_mean_30d' in df.columns:
        df['si_sales_ratio_7_30'] = df['si_sales_mean_7d'] / (df['si_sales_mean_30d'] + 1)
    
    return df

## 7. Execute Feature Engineering Pipeline

In [11]:
# Define date ranges based on winner's solution
train_dates_1 = pd.date_range(Config.TRAIN_START_1, Config.TRAIN_END, freq='D')
train_dates_2 = pd.date_range(Config.TRAIN_START_2, Config.TRAIN_END, freq='D')
val_dates = pd.date_range(Config.VAL_START, Config.VAL_END, freq='D')
test_dates = pd.date_range(Config.TEST_START, Config.TEST_END, freq='D')

print("Date ranges:")
print(f"  Train Option 1: {train_dates_1[0].date()} to {train_dates_1[-1].date()} ({len(train_dates_1)} days)")
print(f"  Train Option 2: {train_dates_2[0].date()} to {train_dates_2[-1].date()} ({len(train_dates_2)} days)")
print(f"  Validation: {val_dates[0].date()} to {val_dates[-1].date()} ({len(val_dates)} days)")
print(f"  Test: {test_dates[0].date()} to {test_dates[-1].date()} ({len(test_dates)} days)")

Date ranges:
  Train Option 1: 2017-05-31 to 2017-07-19 (50 days)
  Train Option 2: 2017-06-14 to 2017-07-19 (36 days)
  Validation: 2017-07-26 to 2017-08-10 (16 days)
  Test: 2017-08-16 to 2017-08-31 (16 days)


In [12]:
# Create training features (using shorter training window for memory efficiency)
# You can switch to train_dates_1 for the longer window if memory allows
print("\n" + "="*70)
print("🏋️ CREATING TRAINING FEATURES")
print("="*70)

train_features = create_training_features(
    train, test, base_df, 
    train_dates_2,  # Using shorter window (Option 2)
    include_target=True
)

# Add difference features
train_features = add_difference_features(train_features)

print(f"\n✓ Training features shape: {train_features.shape}")
print(f"  Memory: {train_features.memory_usage(deep=True).sum() / 1024**3:.2f} GB")


🏋️ CREATING TRAINING FEATURES

🔧 CREATING FEATURES FOR 36 DATES


Processing dates:   0%|          | 0/36 [00:00<?, ?it/s]


  [1/36] Processing 2017-06-14 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [2/36] Processing 2017-06-15 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [3/36] Processing 2017-06-16 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [4/36] Processing 2017-06-17 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [5/36] Processing 2017-06-18 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [6/36] Processing 2017-06-19 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [7/36] Processing 2017-06-20 00:00:00...
    Computing store_item features...
    Computing item 

In [13]:
# Create validation features
print("\n" + "="*70)
print("🔍 CREATING VALIDATION FEATURES")
print("="*70)

val_features = create_training_features(
    train, test, base_df,
    val_dates,
    include_target=True
)

val_features = add_difference_features(val_features)

print(f"\n✓ Validation features shape: {val_features.shape}")


🔍 CREATING VALIDATION FEATURES

🔧 CREATING FEATURES FOR 16 DATES


Processing dates:   0%|          | 0/16 [00:00<?, ?it/s]


  [1/16] Processing 2017-07-26 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [2/16] Processing 2017-07-27 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [3/16] Processing 2017-07-28 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [4/16] Processing 2017-07-29 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [5/16] Processing 2017-07-30 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [6/16] Processing 2017-07-31 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [7/16] Processing 2017-08-01 00:00:00...
    Computing store_item features...
    Computing item 

In [14]:
# Free up memory before test features
gc.collect()

# Create test features
print("\n" + "="*70)
print("🧪 CREATING TEST FEATURES")
print("="*70)

test_features = create_training_features(
    train, test, base_df,
    test_dates,
    include_target=False
)

test_features = add_difference_features(test_features)

print(f"\n✓ Test features shape: {test_features.shape}")


🧪 CREATING TEST FEATURES

🔧 CREATING FEATURES FOR 16 DATES


Processing dates:   0%|          | 0/16 [00:00<?, ?it/s]


  [1/16] Processing 2017-08-16 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [2/16] Processing 2017-08-17 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [3/16] Processing 2017-08-18 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [4/16] Processing 2017-08-19 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [5/16] Processing 2017-08-20 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [6/16] Processing 2017-08-21 00:00:00...
    Computing store_item features...
    Computing item features...
    Computing store_class features...

  [7/16] Processing 2017-08-22 00:00:00...
    Computing store_item features...
    Computing item 

## 8. Fill Missing Values & Final Processing

In [15]:
def fill_missing_values(df, is_train=True):
    """
    Fill missing values.
    Winner's approach: fill missing with 0
    """
    print("\nFilling missing values...")
    
    # Get numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Remove target from fill if present
    if 'unit_sales' in numeric_cols and is_train:
        numeric_cols.remove('unit_sales')
    
    # Fill with 0 (Winner's approach)
    for col in numeric_cols:
        if df[col].isna().sum() > 0:
            df[col] = df[col].fillna(0)
    
    return df


def get_feature_columns(df):
    """Get list of feature columns (exclude metadata)"""
    exclude_cols = ['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 
                   'type', 'state', 'family']
    
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    return feature_cols

In [16]:
# Fill missing values
train_features = fill_missing_values(train_features, is_train=True)
val_features = fill_missing_values(val_features, is_train=True)
test_features = fill_missing_values(test_features, is_train=False)

# Get feature columns
feature_cols = get_feature_columns(train_features)
print(f"\n✓ Number of features: {len(feature_cols)}")
print(f"\nFeature columns:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")


Filling missing values...

Filling missing values...

Filling missing values...

✓ Number of features: 66

Feature columns:
   1. cluster
   2. class
   3. perishable
   4. store_type
   5. family_encoded
   6. state_encoded
   7. si_sales_mean_1d
   8. si_sales_std_1d
   9. si_sales_median_1d
  10. si_sales_mean_3d
  11. si_sales_std_3d
  12. si_sales_median_3d
  13. si_sales_mean_5d
  14. si_sales_std_5d
  15. si_sales_median_5d
  16. si_sales_mean_7d
  17. si_sales_std_7d
  18. si_sales_median_7d
  19. si_sales_mean_14d
  20. si_sales_std_14d
  21. si_sales_median_14d
  22. si_sales_mean_30d
  23. si_sales_std_30d
  24. si_sales_median_30d
  25. si_sales_mean_60d
  26. si_sales_std_60d
  27. si_sales_median_60d
  28. si_sales_mean_140d
  29. si_sales_std_140d
  30. si_sales_median_140d
  31. si_promo_mean_7d
  32. si_promo_sum_7d
  33. si_promo_mean_14d
  34. si_promo_sum_14d
  35. si_promo_mean_30d
  36. si_promo_sum_30d
  37. si_days_since_last
  38. si_zero_ratio_7d
  39. si_zer

## 9. Save Processed Data

In [17]:
def save_features(df, filename):
    """Save features to parquet file"""
    filepath = Config.OUTPUT_PATH / filename
    df.to_parquet(filepath, index=False, compression='snappy')
    size_mb = filepath.stat().st_size / 1024**2
    print(f"✓ Saved {filename} ({size_mb:.1f} MB)")
    return filepath

In [18]:
# Save all datasets
print("\n" + "="*60)
print("💾 SAVING FEATURE DATASETS")
print("="*60)

save_features(train_features, 'train_features.parquet')
save_features(val_features, 'val_features.parquet')
save_features(test_features, 'test_features.parquet')

# Save feature column list
feature_info = {
    'feature_cols': feature_cols,
    'family_map': family_map,
    'state_map': state_map
}

import pickle
with open(Config.OUTPUT_PATH / 'feature_info.pkl', 'wb') as f:
    pickle.dump(feature_info, f)
print("✓ Saved feature_info.pkl")


💾 SAVING FEATURE DATASETS
✓ Saved train_features.parquet (607.0 MB)
✓ Saved val_features.parquet (269.4 MB)
✓ Saved test_features.parquet (203.6 MB)
✓ Saved feature_info.pkl


## 10. Feature Summary

In [19]:
print("\n" + "="*70)
print("📋 FEATURE ENGINEERING SUMMARY")
print("="*70)

print(f"""
Dataset Sizes:
  Training:   {train_features.shape[0]:,} rows × {train_features.shape[1]} columns
  Validation: {val_features.shape[0]:,} rows × {val_features.shape[1]} columns
  Test:       {test_features.shape[0]:,} rows × {test_features.shape[1]} columns

Feature Categories (based on Winner's solution):
  
  1. BASIC FEATURES:
     - store_nbr, item_nbr (IDs)
     - store_type, cluster, state_encoded
     - family_encoded, class, perishable
     - dayofweek

  2. STORE × ITEM FEATURES (primary key):
     - Sales statistics: mean, std, median for [1,3,5,7,14,30,60,140] days
     - Promotion statistics: mean, sum for [7,14,30] days
     - Days since last sale
     - Zero sales ratio for [7,14,30] days

  3. ITEM FEATURES:
     - Sales statistics: mean, std, median for [7,14,30,60] days

  4. STORE × CLASS FEATURES:
     - Sales statistics: mean, median for [7,14,30] days

  5. DIFFERENCE FEATURES:
     - Sales differences between adjacent windows (7-14, 14-30, 30-60)
     - Sales ratio (7d / 30d)

  Winner's Notes:
  ✗ Holidays - NOT useful
  ✗ cluster × item, store × family - NOT useful
  ✓ Only 2017 data used
  ✓ Gap between public/private LB smaller with 16-day model
""")


📋 FEATURE ENGINEERING SUMMARY

Dataset Sizes:
  Training:   7,819,380 rows × 73 columns
  Validation: 3,475,280 rows × 73 columns
  Test:       3,370,464 rows × 73 columns

Feature Categories (based on Winner's solution):
  
  1. BASIC FEATURES:
     - store_nbr, item_nbr (IDs)
     - store_type, cluster, state_encoded
     - family_encoded, class, perishable
     - dayofweek

  2. STORE × ITEM FEATURES (primary key):
     - Sales statistics: mean, std, median for [1,3,5,7,14,30,60,140] days
     - Promotion statistics: mean, sum for [7,14,30] days
     - Days since last sale
     - Zero sales ratio for [7,14,30] days

  3. ITEM FEATURES:
     - Sales statistics: mean, std, median for [7,14,30,60] days

  4. STORE × CLASS FEATURES:
     - Sales statistics: mean, median for [7,14,30] days

  5. DIFFERENCE FEATURES:
     - Sales differences between adjacent windows (7-14, 14-30, 30-60)
     - Sales ratio (7d / 30d)

  Winner's Notes:
  ✗ Holidays - NOT useful
  ✗ cluster × item, store ×

In [20]:
# Display sample features
print("\n--- Sample Training Features ---")
print(train_features.head())

print("\n--- Feature Statistics ---")
print(train_features[feature_cols[:10]].describe())


--- Sample Training Features ---
   store_nbr  item_nbr type  cluster        state        family  class  \
0         25     99197    D        1  Santa Elena     GROCERY I   1067   
1         25    103665    D        1  Santa Elena  BREAD/BAKERY   2712   
2         25    105574    D        1  Santa Elena     GROCERY I   1045   
3         25    105857    D        1  Santa Elena     GROCERY I   1092   
4         25    106716    D        1  Santa Elena     GROCERY I   1032   

   perishable  store_type  family_encoded  ...       date  dayofweek  \
0           0           3               0  ... 2017-06-14          2   
1           1           3               1  ... 2017-06-14          2   
2           0           3               0  ... 2017-06-14          2   
3           0           3               0  ... 2017-06-14          2   
4           0           3               0  ... 2017-06-14          2   

   unit_sales  onpromotion  si_sales_diff_7_14  si_sales_diff_14_30  \
0         0.0    

## 11. Clean Up

In [21]:
# Clean up to free memory for model training
del train
gc.collect()

print("\n✓ Memory cleaned. Ready for model training!")
print(f"\nSaved files in {Config.OUTPUT_PATH}:")
for f in Config.OUTPUT_PATH.glob('*.parquet'):
    print(f"  • {f.name}")
for f in Config.OUTPUT_PATH.glob('*.pkl'):
    print(f"  • {f.name}")


✓ Memory cleaned. Ready for model training!

Saved files in /kaggle/working:
  • val_features.parquet
  • test_features.parquet
  • train_features.parquet
  • feature_info.pkl


---
## Next Steps

1. **Model Training**: Use LightGBM/XGBoost with the created features
2. **Multiple Models**: Train different models with different date ranges
3. **Ensemble**: Combine predictions from multiple models
4. **Evaluation**: Use NWRMSLE metric